In [15]:
import sys
import os
import pandas as pd
import common.processing as processing

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

print("The project base is " + project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

from common.common import display_df, init_notebook


init_notebook()

The project base is /Users/seanxiao/Projects/scorecardpipeline/data


{'td': [('background-color', '#f0f0f0'), ('color', '#333333')],
 'tr:hover': [('background-color', '#ffffb3')],
 'th': [('color', 'white'), ('background-color', '#000066')]}

In [47]:
#load the data
loanData = pd.read_csv("../data/loan_data_2015.csv", sep=",")

/var/folders/f4/8zd2hqxj7ggg05_ks81jr3kw0000gn/T/ipykernel_96523/2273342807.py:2: DtypeWarning: Columns (19,47,55) have mixed types. Specify dtype option on import or set low_memory=False.
  loanData = pd.read_csv("../data/loan_data_2015.csv", sep=",")


In [48]:
pd.set_option('display.max_rows', None)  # Display all rows
pd.set_option('display.max_columns', None)
pd.options.display.max_columns = None
loanData.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421094 entries, 0 to 421093
Data columns (total 74 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   gid                          421094 non-null  int64  
 1   member_id                    421094 non-null  int64  
 2   loan_amnt                    421094 non-null  int64  
 3   funded_amnt                  421094 non-null  int64  
 4   funded_amnt_inv              421094 non-null  int64  
 5   term                         421094 non-null  object 
 6   int_rate                     421094 non-null  float64
 7   installment                  421094 non-null  float64
 8   grade                        421094 non-null  object 
 9   sub_grade                    421094 non-null  object 
 10  emp_title                    397220 non-null  object 
 11  emp_length                   397277 non-null  object 
 12  home_ownership               421094 non-null  object 
 13 

In [57]:
from common import datadictionary
import importlib
importlib.reload(datadictionary)
dictionary = datadictionary.DataDictionary()

dictionary.generate_dataframe(loanData)
display_df(dictionary.generated_dictionary)
dictionary.save_generated_dictionary("../data/loan/data_dictionary.xlsx")


TypeError: DataDictionary.__init__() missing 1 required positional argument: 'default_dictionary_location'

In [58]:
import pandas as pd
import numpy as np

def score_breakdown(df, score_col, bad_col='bad', pdo=50):

    # Validate inputs
    # Filter out missing scores
    clean_df = df[[score_col, bad_col]].dropna(subset=[score_col])
    
    # Create score bins
    min_score = np.floor(clean_df[score_col].min() / pdo) * pdo
    max_score = np.ceil(clean_df[score_col].max() / pdo) * pdo
    bins = np.arange(min_score, max_score + pdo, pdo)
    
    # Create labels for bins
    labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
    
    # Create bins and calculate metrics
    clean_df['Score Range'] = pd.cut(clean_df[score_col], bins=bins, labels=labels, include_lowest=True)
    
    # Group by score range
    breakdown = clean_df.groupby('Score Range', observed=True).agg(
        Total_Count=(bad_col, 'count'),
        Bad_Count=(bad_col, 'sum')
    ).reset_index()
    

    breakdown['Bad_Rate'] = breakdown['Bad_Count'] / breakdown['Total_Count']
    

    full_range = pd.DataFrame({'Score Range': labels})
    breakdown = full_range.merge(breakdown, how='left').fillna(0)

    breakdown.columns = ['Score Range', 'Total Count', 'Bad Count', 'Bad Rate']
    
    return breakdown

# Example usage
if __name__ == "__main__":
    # Create sample data
    np.random.seed(42)
    data = pd.DataFrame({
        'score': np.random.randint(300, 800, 1000),
        'bad': np.random.choice([0, 1], 1000, p=[0.85, 0.15])
    })
    
    # Generate breakdown
    result = score_breakdown(data, 'score', pdo=50)
    print(result)

  Score Range  Total Count  Bad Count  Bad Rate
0     300-350           93         11      0.12
1     350-400           93         11      0.12
2     400-450          107         17      0.16
3     450-500          112         18      0.16
4     500-550          100         17      0.17
5     550-600           99         13      0.13
6     600-650           98         14      0.14
7     650-700          107         14      0.13
8     700-750           96         21      0.22
9     750-800           95         21      0.22


In [ ]:
select = processing.FeatureSelection(target=target, engine="toad", identical=0.95, empty=0.95, iv=0.02, corr=0.6)
select.fit(train)

train_select = select.transform(train)
test_select = select.transform(test)